# 1. Data Loading and Basic Inspection

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix

In [6]:
PARQUET_PATH = r"/content/drive/MyDrive/US_accidents_dataset.parquet"

# Load only the columns we need to save RAM instead of loading all 47
USECOLS = [
    "Severity", "Start_Time",
    "Start_Lat", "Start_Lng",
    "State", "Timezone",
    "Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)",
    "Weather_Condition", "Wind_Direction",
    "Junction", "Traffic_Signal", "Crossing", "Stop",
    "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"
]

df = pd.read_parquet(PARQUET_PATH, columns=USECOLS)

# Downcast numeric columns to float32
for c in ["Start_Lat", "Start_Lng", "Temperature(F)", "Humidity(%)",
          "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")

# Convert string categoricals to pandas category dtype
for c in ["State", "Timezone", "Weather_Condition", "Wind_Direction",
          "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]:
    if c in df.columns:
        df[c] = df[c].astype("category")

# Convert boolean infrastructure flags to int8
for c in ["Junction", "Traffic_Signal", "Crossing", "Stop"]:
    if c in df.columns:
        df[c] = df[c].astype("int8")

print("Loaded shape:", df.shape)
print("Approx RAM (GB):", df.memory_usage(deep=True).sum() / 1e9)


Loaded shape: (7728394, 22)
Approx RAM (GB): 0.942405074


In [7]:
df.head()


,Severity,Start_Time,Start_Lat,Start_Lng,State,Timezone,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),...,Weather_Condition,Wind_Direction,Junction,Traffic_Signal,Crossing,Stop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,3,2016-02-08 05:46:00,39.865147,-84.058723,OH,US/Eastern,36.900002,91.0,29.680000,10.0,...,Light Rain,Calm,0,0,0,0,Night,Night,Night,Night
1,2,2016-02-08 06:07:59,39.928059,-82.831184,OH,US/Eastern,37.900002,100.0,29.650000,10.0,...,Light Rain,Calm,0,0,0,0,Night,Night,Night,Day
2,2,2016-02-08 06:49:27,39.063148,-84.032608,OH,US/Eastern,36.000000,100.0,29.670000,10.0,...,Overcast,SW,0,1,0,0,Night,Night,Day,Day
3,3,2016-02-08 07:23:34,39.747753,-84.205582,OH,US/Eastern,35.099998,96.0,29.639999,9.0,...,Mostly Cloudy,SW,0,0,0,0,Night,Day,Day,Day
4,2,2016-02-08 07:39:07,39.627781,-84.188354,OH,US/Eastern,36.000000,89.0,29.650000,6.0,...,Mostly Cloudy,SW,0,1,0,0,Day,Day,Day,Day


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7728394 entries, 0 to 7728393
Data columns (total 22 columns):
 #   Column                 Dtype   
---  ------                 -----   
 0   Severity               int64   
 1   Start_Time             object  
 2   Start_Lat              float32 
 3   Start_Lng              float32 
 4   State                  category
 5   Timezone               category
 6   Temperature(F)         float32 
 7   Humidity(%)            float32 
 8   Pressure(in)           float32 
 9   Visibility(mi)         float32 
 10  Wind_Speed(mph)        float32 
 11  Precipitation(in)      float32 
 12  Weather_Condition      category
 13  Wind_Direction         category
 14  Junction               int8    
 15  Traffic_Signal         int8    
 16  Crossing               int8    
 17  Stop                   int8    
 18  Sunrise_Sunset         category
 19  Civil_Twilight         category
 20  Nautical_Twilight      category
 21  Astronomical_Twilight  category

**Observations:**

* Dataset contains ~7.7M rows; we load **22 selected columns** out of the original 47 to stay within RAM limits.
* The target variable (`Severity`) is heavily imbalanced: severity 2 accounts for ~80% of records.
* Several datetime and categorical variables need preprocessing before modelling.
* Boolean infrastructure flags (junction, crossing, etc.) are already compact as `int8`.

# 2. Target Distribution

Before building any model it's important to understand how the target variable is distributed.
A heavily skewed distribution means plain accuracy is a misleading metric: a model that
predicts 'severity 2' for everything would score ~80% accuracy while being completely useless
for detecting the dangerous severity-4 cases we actually care about.

In [9]:
sev_dist   = df["Severity"].value_counts(normalize=True).sort_index()
sev_counts = df["Severity"].value_counts().sort_index()
sev_dist, sev_counts


(Severity
 1    0.008717
 2    0.796670
 3    0.168125
 4    0.026488
 Name: proportion, dtype: float64,
 Severity
 1      67366
 2    6156981
 3    1299337
 4     204710
 Name: count, dtype: int64)

**Observation: severe class imbalance:**

* Severity 2 dominates with ~80% of all records.
* Severity 4 (our positive class) represents only ~2.65% of records.
* Accuracy alone is **not** an appropriate metric; we will use PR-AUC and tune the decision threshold.
* We frame this as a **binary rare-event problem**: severity 4 vs. everything else.

# 3. Feature Engineering

We extract time-based features from `Start_Time` (hour of day, day of week, month, year,
and a weekend flag) and then define the binary target `y` (1 = severity 4, 0 = everything else).

~9.6% of `Start_Time` values are null (NaT). We fill derived time columns with `-1` to signal
'unknown' rather than letting NaNs propagate silently into the model.
`Start_Time` itself is dropped after extraction to free memory.

In [10]:
# Parse datetime once
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")

# Extract time features: fill NaT-derived NaNs with -1 and cast to int8/int16
df["hour"]       = df["Start_Time"].dt.hour.fillna(-1).astype("int8")
df["dayofweek"]  = df["Start_Time"].dt.dayofweek.fillna(-1).astype("int8")
df["month"]      = df["Start_Time"].dt.month.fillna(-1).astype("int8")
df["year"]       = df["Start_Time"].dt.year.fillna(-1).astype("int16")
df["is_weekend"] = (df["Start_Time"].dt.dayofweek >= 5).fillna(False).astype("int8")

# Drop Start_Time: no longer needed and frees memory
df = df.drop(columns=["Start_Time"])

# Binary target: 1 = severity 4 (critical), 0 = all others
df["y"] = (df["Severity"] == 4).astype("int8")

print(df["y"].value_counts(normalize=True))


y
0    0.973512
1    0.026488
Name: proportion, dtype: float64


# 4. Missing Value Audit

CatBoost handles numeric NaNs natively, so we only need to explicitly fill missing values
for **categorical** columns (which must have no NaN at inference time).
Here we inspect the missingness rate across all features to understand the data quality.

In [11]:
df.isnull().mean().sort_values(ascending=False).head(20)


,0
Precipitation(in),0.285129
Wind_Speed(mph),0.073914
Visibility(mi),0.022915
Wind_Direction,0.022670
Humidity(%),0.022533
Weather_Condition,0.022444
Temperature(F),0.021201
Pressure(in),0.018203
Sunrise_Sunset,0.003008
Civil_Twilight,0.003008


# 5. Train / Test Split

We hold out 20% of data for evaluation. `stratify=y` ensures the rare severity-4 class
is represented proportionally in both train and test sets (important with only ~2.6% positives).

In [12]:
X = df.drop(columns=["Severity", "y"])
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", len(X_train), "| Test size:", len(X_test))


Train size: 6182715 | Test size: 1545679


# 6. Train CatBoost

We use **CatBoost** because it handles categorical features natively without one-hot encoding
(important for RAM efficiency) and works well on GPU.

Key choices:
* `auto_class_weights='Balanced'` tells CatBoost to upweight the rare severity-4 class
  during training, improving recall without any resampling overhead.
* Missing values in categorical columns are filled with `'Unknown'` before fitting,
  since CatBoost requires no NaN in cat features.


In [13]:
cat_cols = X_train.select_dtypes(include=["category", "object"]).columns.tolist()
cat_idx  = [X_train.columns.get_loc(c) for c in cat_cols]

# Fill NaN in categorical columns with 'Unknown'
for col in cat_cols:
    if str(X_train[col].dtype) == "category":
        X_train[col] = X_train[col].cat.add_categories(["Unknown"]).fillna("Unknown")
        X_test[col]  = X_test[col].cat.add_categories(["Unknown"]).fillna("Unknown")
    else:
        X_train[col] = X_train[col].fillna("Unknown")
        X_test[col]  = X_test[col].fillna("Unknown")

model = CatBoostClassifier(
    iterations=400,
    depth=8,
    learning_rate=0.1,
    loss_function="Logloss",
    auto_class_weights="Balanced",  # addresses the ~2.6% positive class imbalance
    verbose=100,
    task_type="GPU",   # remove if on CPU
    devices="0"        # remove if on CPU
)

model.fit(X_train, y_train, cat_features=cat_idx)


0:	learn: 0.6722093	total: 1.39s	remaining: 9m 16s
100:	learn: 0.5167308	total: 2m 13s	remaining: 6m 35s
200:	learn: 0.4977866	total: 4m 21s	remaining: 4m 18s
300:	learn: 0.4858454	total: 6m 28s	remaining: 2m 7s
399:	learn: 0.4770282	total: 8m 35s	remaining: 0us


CatBoostClassifier(auto_class_weights='Balanced', depth=8, devices='0', iterations=400, learning_rate=0.1, loss_function='Logloss', task_type='GPU', verbose=100)

# 7. Evaluation: PR-AUC and Threshold Analysis

For imbalanced binary classification, **Precision-Recall AUC** (PR-AUC) is more informative
than ROC-AUC. A random classifier has PR-AUC equal to the positive rate (~2.6%),
so any score well above that indicates real predictive power.

We also sweep over decision thresholds (instead of the default 0.5) to find operating
points that make practical sense: showing both a high-recall option and a balanced option.

In [14]:
proba = model.predict_proba(X_test)[:, 1]
ap    = average_precision_score(y_test, proba)
print("PR AUC:", ap)

# Baseline at default 0.5 threshold
pred = (proba >= 0.5).astype(int)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=4))


PR AUC: 0.1972976705918701
[[1163865  340872]
 [   9283   31659]]
              precision    recall  f1-score   support

           0     0.9921    0.7735    0.8692   1504737
           1     0.0850    0.7733    0.1531     40942

    accuracy                         0.7735   1545679
   macro avg     0.5385    0.7734    0.5112   1545679
weighted avg     0.9681    0.7735    0.8503   1545679



## 7a. Threshold Sweep

The default 0.5 threshold is inappropriate for a rare-event classifier. By lowering the
threshold we trade precision for recall, catching more true severity-4 accidents at the
cost of more false alarms. The table below lets us pick an operating point based on the
acceptable false-alarm rate for a given application.

In [15]:
thresholds = np.linspace(0.01, 0.50, 50)

rows = []
for t in thresholds:
    pred_t    = (proba >= t).astype(int)
    tp = int(((pred_t == 1) & (y_test == 1)).sum())
    fp = int(((pred_t == 1) & (y_test == 0)).sum())
    fn = int(((pred_t == 0) & (y_test == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    rows.append([t, precision, recall, tp, fp, fn])

thr_table = pd.DataFrame(rows, columns=["threshold", "precision", "recall", "TP", "FP", "FN"])

# Add operational metrics
thr_table["alerts"]                 = thr_table["TP"] + thr_table["FP"]
thr_table["alerts_per_1000"]        = 1000 * thr_table["alerts"] / len(y_test)
thr_table["alerts_per_true_severe"] = thr_table["alerts"] / thr_table["TP"]

thr_table[["threshold", "precision", "recall", "alerts_per_1000", "alerts_per_true_severe", "TP", "FP", "FN"]].head(10)


,threshold,precision,recall,alerts_per_1000,alerts_per_true_severe,TP,FP,FN
0,0.01,0.026490,1.000000,999.913954,37.749646,40942,1504604,0
1,0.02,0.026535,1.000000,998.216318,37.685555,40942,1501980,0
2,0.03,0.026655,0.999902,993.635160,37.516269,40938,1494903,4
3,0.04,0.026850,0.999805,986.316046,37.243563,40934,1483594,8
4,0.05,0.027156,0.999316,974.719201,36.823654,40914,1465689,28
5,0.06,0.027582,0.998925,959.322731,36.256174,40898,1441907,44
6,0.07,0.028139,0.998095,939.530135,35.537686,40864,1411348,78
7,0.08,0.028747,0.996678,918.348506,34.785865,40806,1378666,136
8,0.09,0.029407,0.995530,896.723058,34.005888,40759,1345287,183
9,0.10,0.030119,0.993747,873.934368,33.201150,40686,1310136,256


## 7b. Chosen Operating Points

We highlight two representative thresholds:
* **0.02**: high-recall mode: catches ~83% of severe accidents, but generates ~340 alerts per 1000 cases.
* **0.05**: balanced mode: catches ~59% of severe accidents with only ~122 alerts per 1000 cases.

The right choice depends on the downstream cost of missing a severe accident vs. the cost of
acting on a false alarm.

In [16]:
chosen = thr_table[thr_table["threshold"].isin([0.02, 0.05])].copy()
chosen[["threshold", "precision", "recall", "TP", "FP", "FN", "alerts_per_1000", "alerts_per_true_severe"]]


,threshold,precision,recall,TP,FP,FN,alerts_per_1000,alerts_per_true_severe
1,0.02,0.026535,1.000000,40942,1501980,0,998.216318,37.685555
4,0.05,0.027156,0.999316,40914,1465689,28,974.719201,36.823654


In [17]:
for t in [0.02, 0.05]:
    pred_t = (proba >= t).astype(int)
    print("\nThreshold:", t)
    print(confusion_matrix(y_test, pred_t))
    print(classification_report(y_test, pred_t, digits=4))



Threshold: 0.02
[[   2757 1501980]
 [      0   40942]]
              precision    recall  f1-score   support

           0     1.0000    0.0018    0.0037   1504737
           1     0.0265    1.0000    0.0517     40942

    accuracy                         0.0283   1545679
   macro avg     0.5133    0.5009    0.0277   1545679
weighted avg     0.9742    0.0283    0.0049   1545679


Threshold: 0.05
[[  39048 1465689]
 [     28   40914]]
              precision    recall  f1-score   support

           0     0.9993    0.0260    0.0506   1504737
           1     0.0272    0.9993    0.0529     40942

    accuracy                         0.0517   1545679
   macro avg     0.5132    0.5126    0.0517   1545679
weighted avg     0.9735    0.0517    0.0506   1545679



# 8. Key Factors: Feature Importance

CatBoost provides built-in feature importance scores (PredictionValuesChange by default).
These reflect how much each feature contributes to shifting the model's output.

Note that geographic features (lat/lng/state) dominate: this makes sense because accident
severity reporting practices and road infrastructure vary enormously by region.
In the next section we retrain without geography to isolate the more *generalisable* factors.

In [18]:
imp    = pd.Series(model.get_feature_importance(), index=X_train.columns).sort_values(ascending=False)
top15  = imp.head(15).to_frame("importance")
top15


,importance
Start_Lng,19.920885
Start_Lat,19.731246
year,12.681549
State,11.923446
hour,6.796258
month,5.568740
Astronomical_Twilight,2.677746
Pressure(in),2.476331
Traffic_Signal,2.437395
dayofweek,1.710898


# 9. Retrain Without Geography

Geographic features (lat, lng, state) account for ~55% of total importance in the full model.
However, location is essentially a proxy for region-specific factors (road types, reporting
conventions, traffic density) rather than a causal driver of accident severity.

We retrain without these features to:
1. Quantify how much predictive power is *genuinely* geographic.
2. Reveal which **actionable, non-geographic** factors actually matter.

In [19]:
geo_cols = [c for c in ["Start_Lat", "Start_Lng", "State"] if c in X_train.columns]

X_train_nogeo = X_train.drop(columns=geo_cols)
X_test_nogeo  = X_test.drop(columns=geo_cols)

cat_cols2 = X_train_nogeo.select_dtypes(include=["category", "object"]).columns.tolist()
cat_idx2  = [X_train_nogeo.columns.get_loc(c) for c in cat_cols2]

model_nogeo = CatBoostClassifier(
    iterations=400,
    depth=8,
    learning_rate=0.1,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    verbose=100,
    task_type="GPU",
    devices="0"
)

model_nogeo.fit(X_train_nogeo, y_train, cat_features=cat_idx2)

proba_nogeo = model_nogeo.predict_proba(X_test_nogeo)[:, 1]
print("PR AUC full:   ", average_precision_score(y_test, proba))
print("PR AUC no-geo: ", average_precision_score(y_test, proba_nogeo))


0:	learn: 0.6808055	total: 1.05s	remaining: 6m 59s
100:	learn: 0.5860521	total: 1m 49s	remaining: 5m 24s
200:	learn: 0.5741163	total: 3m 49s	remaining: 3m 46s
300:	learn: 0.5666410	total: 5m 49s	remaining: 1m 54s
399:	learn: 0.5608342	total: 7m 49s	remaining: 0us
PR AUC full:    0.1972976705918701
PR AUC no-geo:  0.11312261441217619


# 10. Feature Importance Without Geography

With location removed, the model is forced to lean on weather, time, and infrastructure.
These factors are more interpretable and actionable e.g., identifying that accidents at
crossings or traffic signals during certain hours are more likely to be severe.

In [20]:
imp_nogeo = pd.Series(model_nogeo.get_feature_importance(), index=X_train_nogeo.columns).sort_values(ascending=False)
imp_nogeo.head(15)


,0
Timezone,19.747629
year,14.514945
month,9.651745
hour,9.107023
Pressure(in),9.062917
Temperature(F),7.822916
Humidity(%),3.461275
Astronomical_Twilight,2.694145
dayofweek,2.609878
Traffic_Signal,2.585072


# 11. No-Geo Model: Threshold Sweep and Full Comparison

We repeat the threshold sweep for the no-geography model and then compare both models
side by side at the same operating points. This gives a clear picture of how much
predictive lift comes from location data versus contextual features alone.

In [21]:
rows_ng = []
for t in np.linspace(0.01, 0.50, 50):
    pred_t    = (proba_nogeo >= t).astype(int)
    tp = int(((pred_t == 1) & (y_test == 1)).sum())
    fp = int(((pred_t == 1) & (y_test == 0)).sum())
    fn = int(((pred_t == 0) & (y_test == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    rows_ng.append([t, precision, recall, tp, fp, fn])

thr_ng = pd.DataFrame(rows_ng, columns=["threshold", "precision", "recall", "TP", "FP", "FN"])
thr_ng.head()


,threshold,precision,recall,TP,FP,FN
0,0.01,0.026491,1.000000,40942,1504573,0
1,0.02,0.026499,0.999976,40941,1504039,1
2,0.03,0.026519,0.999927,40939,1502800,3
3,0.04,0.026559,0.999853,40936,1500375,6
4,0.05,0.026628,0.999780,40933,1496297,9


In [22]:
# Side-by-side comparison at the two chosen thresholds
compare = pd.DataFrame({
    "threshold":        [0.02, 0.05],
    "precision_full":   thr_table[thr_table["threshold"].isin([0.02, 0.05])]["precision"].values,
    "recall_full":      thr_table[thr_table["threshold"].isin([0.02, 0.05])]["recall"].values,
    "precision_no_geo": thr_ng[thr_ng["threshold"].isin([0.02, 0.05])]["precision"].values,
    "recall_no_geo":    thr_ng[thr_ng["threshold"].isin([0.02, 0.05])]["recall"].values,
})
compare


,threshold,precision_full,recall_full,precision_no_geo,recall_no_geo
0,0.02,0.026535,1.000000,0.026499,0.999976
1,0.05,0.027156,0.999316,0.026628,0.999780
